<a href="https://colab.research.google.com/github/pramodkumarw/Github-Colab/blob/main/checkpointer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Langgraph

In [ ]:
# STEP 1: SET UP THE ENVIRONMENT

# STEP 1.1 INSTALL THE REQUIRED PACKAGES
!pip install langchain_community

!pip install langchain-groq
!pip install langchain
!pip install langgrah

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
# from rich import print
from google.colab import userdata

In [ ]:

GROQ_API_KEY=userdata.get("GROQ_API_KEY")
llm=ChatGroq(model="openai/gpt-oss-120b", groq_api_key=GROQ_API_KEY)


In [ ]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str


In [ ]:
def generate_joke(state: JokeState):
    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content
    return {'joke': response}

def generate_explanation(state: JokeState):
    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content
    return {'explanation': response}

In [ ]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

In [ ]:
workflow.get_state(config1)

In [ ]:
list(workflow.get_state_history(config1))